# Load WLASL Dataset with FiftyOne

This notebook loads the **WLASL (Word-Level American Sign Language)** dataset from the FiftyOne / Hugging Face hub, saves it locally, checks whether the video files exist on your machine, logs the process, and finally opens the FiftyOne visual app so you can inspect the dataset.

We are doing it slowly and step by step so that each part is easy to understand.

## 1. Import basic Python libraries

These libraries help us work with files, folders, logs, system output, and timestamps.

- `os` and `sys`: system-level utilities
- `logging`: records what the script is doing
- `Path`: easier file and folder path handling
- `datetime`: creates timestamped log filenames

In [9]:
import os
import sys
import logging
from pathlib import Path
from datetime import datetime

## 2. Create project folders

Here we define a main project folder called `ksl_project_data`.

Inside it, we also create a `logs` folder where log files will be stored.

The option `parents=True` means Python will create parent folders if they do not exist.  
The option `exist_ok=True` means Python will not raise an error if the folder already exists.

In [10]:
PROJECT_DIR = Path("ksl_project_data").resolve()
LOG_DIR = PROJECT_DIR / "logs"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Log folder:", LOG_DIR)

Project folder: /Users/lyrics/Desktop/client/evance ai project/dataset/ksl_project_data
Log folder: /Users/lyrics/Desktop/client/evance ai project/dataset/ksl_project_data/logs


## 3. Configure logging

Logging helps us track what happened when the notebook runs.

This creates a log file with the current date and time, for example:

`load_wlasl_20260604_143000.log`

Logs are written to two places:

1. A log file inside `ksl_project_data/logs`
2. The notebook output screen

In [11]:
log_file = LOG_DIR / f"load_wlasl_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler(sys.stdout),
    ],
    force=True,  # important in notebooks so logging can be reconfigured
)

logger = logging.getLogger(__name__)

logger.info("Logging configured successfully")
logger.info(f"Log file location: {log_file}")

2026-06-04 12:26:01,232 | INFO | Logging configured successfully
2026-06-04 12:26:01,234 | INFO | Log file location: /Users/lyrics/Desktop/client/evance ai project/dataset/ksl_project_data/logs/load_wlasl_20260604_122601.log


## 4. Import FiftyOne libraries

FiftyOne is used for dataset management and visual inspection.

We import:

- `fiftyone as fo`: main FiftyOne library
- `fiftyone.utils.huggingface as fouh`: helper utilities for loading datasets from Hugging Face

In [14]:
import fiftyone as fo
import fiftyone.utils.huggingface as fouh

logger.info("FiftyOne imported successfully")

DATASET_NAME = "Voxel51/WLASL"

logger.info("Checking if WLASL already exists locally...")

if DATASET_NAME in fo.list_datasets():
    logger.info("WLASL already exists locally. Loading existing dataset...")
    dataset = fo.load_dataset(DATASET_NAME)
else:
    logger.info("Loading WLASL from Hugging Face/FiftyOne hub...")
    dataset = fouh.load_from_hub(DATASET_NAME)
    dataset.persistent = True

logger.info(f"Dataset loaded: {dataset.name}")
logger.info(f"Total samples in metadata: {len(dataset)}")

print("Dataset name:", dataset.name)
print("Total samples:", len(dataset))

2026-06-04 12:29:42,529 | INFO | FiftyOne imported successfully
2026-06-04 12:29:42,531 | INFO | Checking if WLASL already exists locally...
2026-06-04 12:29:42,543 | INFO | WLASL already exists locally. Loading existing dataset...
2026-06-04 12:29:42,618 | INFO | Dataset loaded: Voxel51/WLASL
2026-06-04 12:29:42,622 | INFO | Total samples in metadata: 11980
Dataset name: Voxel51/WLASL
Total samples: 11980


## 5. Load the WLASL dataset

This cell loads the dataset named `Voxel51/WLASL` from the Hugging Face / FiftyOne hub.

WLASL means **Word-Level American Sign Language**.

Setting `dataset.persistent = True` means the dataset remains saved in your local FiftyOne database, so you do not have to reload everything from scratch every time.

In [15]:
logger.info("Loading WLASL from Hugging Face/FiftyOne cache...")

dataset = fouh.load_from_hub("Voxel51/WLASL")
dataset.persistent = True

logger.info(f"Dataset loaded: {dataset.name}")
logger.info(f"Total samples in metadata: {len(dataset)}")

print("Dataset name:", dataset.name)
print("Total samples:", len(dataset))

2026-06-04 12:30:14,979 | INFO | Loading WLASL from Hugging Face/FiftyOne cache...
2026-06-04 12:30:15,865 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/Voxel51/WLASL/resolve/main/fiftyone.yml "HTTP/1.1 307 Temporary Redirect"
2026-06-04 12:30:15,878 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Voxel51/WLASL/3cf8daaac08088798f539d62fa511028bf5e6fd0/fiftyone.yml "HTTP/1.1 200 OK"
2026-06-04 12:30:15,880 | INFO | Downloading config file fiftyone.yml from Voxel51/WLASL
2026-06-04 12:30:16,116 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/Voxel51/WLASL/resolve/main/fiftyone.yml "HTTP/1.1 307 Temporary Redirect"
2026-06-04 12:30:16,137 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Voxel51/WLASL/3cf8daaac08088798f539d62fa511028bf5e6fd0/fiftyone.yml "HTTP/1.1 200 OK"
Loading dataset
2026-06-04 12:30:16,141 | INFO | Loading dataset
2026-06-04 12:30:16,396 | INFO | HTTP Request: GET https://huggingface.

ValueError: Dataset name 'Voxel51/WLASL' is not available

## 6. Check whether video files exist locally

The dataset metadata may load successfully, but some actual video files may not exist locally yet.

This cell checks every sample in the dataset:

- If the `sample.filepath` exists on the machine, it counts as `existing`
- If the file does not exist, it counts as `missing`
- The notebook stores up to 20 missing file examples for debugging

In [16]:
existing = 0
missing = 0
missing_examples = []

for sample in dataset:
    if sample.filepath and Path(sample.filepath).exists():
        existing += 1
    else:
        missing += 1
        if len(missing_examples) < 20:
            missing_examples.append(sample.filepath)

logger.info(f"Existing local video files: {existing}")
logger.info(f"Missing local video files: {missing}")

print("Existing local video files:", existing)
print("Missing local video files:", missing)

2026-06-04 12:31:11,676 | INFO | Existing local video files: 11880
2026-06-04 12:31:11,684 | INFO | Missing local video files: 100
Existing local video files: 11880
Missing local video files: 100


## 7. Display missing video examples

If there are missing video files, this cell prints a few example paths.

This is useful because sometimes the dataset metadata exists, but the video files have not been downloaded correctly or the file paths point to a missing cache location.

In [17]:
if missing_examples:
    logger.warning("Missing video examples:")
    print("Missing video examples:")
    
    for path in missing_examples:
        logger.warning(path)
        print(path)
else:
    logger.info("No missing video examples found")
    print("No missing video examples found")

2026-06-04 12:32:28,641 | WARNING | Missing video examples:
Missing video examples:
2026-06-04 12:32:28,645 | WARNING | /Users/lyrics/fiftyone/huggingface/hub/Voxel51/WLASL/data/data_18/10450.mp4
/Users/lyrics/fiftyone/huggingface/hub/Voxel51/WLASL/data/data_18/10450.mp4
2026-06-04 12:32:28,646 | WARNING | /Users/lyrics/fiftyone/huggingface/hub/Voxel51/WLASL/data/data_18/10453.mp4
/Users/lyrics/fiftyone/huggingface/hub/Voxel51/WLASL/data/data_18/10453.mp4
2026-06-04 12:32:28,648 | WARNING | /Users/lyrics/fiftyone/huggingface/hub/Voxel51/WLASL/data/data_18/10461.mp4
/Users/lyrics/fiftyone/huggingface/hub/Voxel51/WLASL/data/data_18/10461.mp4
2026-06-04 12:32:28,649 | WARNING | /Users/lyrics/fiftyone/huggingface/hub/Voxel51/WLASL/data/data_18/10462.mp4
/Users/lyrics/fiftyone/huggingface/hub/Voxel51/WLASL/data/data_18/10462.mp4
2026-06-04 12:32:28,651 | WARNING | /Users/lyrics/fiftyone/huggingface/hub/Voxel51/WLASL/data/data_18/10464.mp4
/Users/lyrics/fiftyone/huggingface/hub/Voxel51/WLASL

## 8. Optional: View a few dataset samples

Before launching the full FiftyOne app, we can quickly inspect a few samples directly in the notebook.

This helps confirm that the dataset has file paths and sample metadata.

In [18]:
for sample in dataset.take(5):
    print("Sample ID:", sample.id)
    print("File path:", sample.filepath)
    print("Fields:", sample.field_names)
    print("-" * 80)

Sample ID: 652ec7727d9a3b27226e254a
File path: /Users/lyrics/fiftyone/huggingface/hub/Voxel51/WLASL/data/data_44/26525.mp4
Fields: ('id', 'filepath', 'tags', 'metadata', 'created_at', 'last_modified_at', 'bounding_box', 'gloss')
--------------------------------------------------------------------------------
Sample ID: 652ec7727d9a3b27226e31e8
File path: /Users/lyrics/fiftyone/huggingface/hub/Voxel51/WLASL/data/data_77/47615.mp4
Fields: ('id', 'filepath', 'tags', 'metadata', 'created_at', 'last_modified_at', 'bounding_box', 'gloss')
--------------------------------------------------------------------------------
Sample ID: 652ec7727d9a3b27226e2237
File path: /Users/lyrics/fiftyone/huggingface/hub/Voxel51/WLASL/data/data_36/21281.mp4
Fields: ('id', 'filepath', 'tags', 'metadata', 'created_at', 'last_modified_at', 'bounding_box', 'gloss')
--------------------------------------------------------------------------------
Sample ID: 652ec7727d9a3b27226e213b
File path: /Users/lyrics/fiftyone/

## 9. Launch the FiftyOne app

This opens the FiftyOne web interface where you can visually inspect the WLASL dataset.

In a local machine, this usually opens in your browser.

The `session.wait()` line keeps the app running until you stop it manually.

In [19]:
logger.info("Launching FiftyOne app...")

session = fo.launch_app(dataset)
session.wait()

2026-06-04 12:33:30,492 | INFO | Launching FiftyOne app...

Could not connect session, trying again in 10 seconds



RuntimeError: Client is not connected